# SMS Chat Analysis


## Import and look over your dataset

In [1]:
import pandas as pd

# Load the SMS dataset into a pandas DataFrame
df = pd.read_csv("clean_nus_sms.csv")

# Display first 5 rows to inspect the data
df.head()

,Unnamed: 0,id,Message,length,country,Date
0,0,10120,Bugis oso near wat...,21,SG,2003/4
1,1,10121,"Go until jurong point, crazy.. Available only ...",111,SG,2003/4
2,2,10122,I dunno until when... Lets go learn pilates...,46,SG,2003/4
3,3,10123,Den only weekdays got special price... Haiz......,140,SG,2003/4
4,4,10124,Meet after lunch la...,22,SG,2003/4


## Project Analysis Plan

This project aims to explore SMS communication patterns using Natural Language Processing techniques.

Planned analyses:

1. Exploratory text analysis
   - Message length distribution
   - Word frequency distribution
   - Common bigrams

2. Sentiment analysis
   - Distribution of positive, negative, and neutral messages
   - Relationship between sentiment and message length

3. Topic modeling
   - Identify common themes using TF-IDF and LDA

4. Text similarity analysis
   - Measure similarity between messages using cosine similarity

5. Part-of-Speech tagging
   - Identify commonly used linguistic patterns

## Data Cleaning

In [2]:
# Check number of rows and columns
df.shape

(48598, 6)

In [3]:
# Check column types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48598 entries, 0 to 48597
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  48598 non-null  int64 
 1   id          48598 non-null  int64 
 2   Message     48595 non-null  object
 3   length      48598 non-null  object
 4   country     48598 non-null  object
 5   Date        48598 non-null  object
dtypes: int64(2), object(4)
memory usage: 2.2+ MB


In [4]:
# Count missing values per column
df.isnull().sum()

Unnamed: 0    0
id            0
Message       3
length        0
country       0
Date          0
dtype: int64

In [5]:
# Count duplicate rows
df.duplicated().sum()

0

In [6]:
# Drop unnecessary index column
df = df.drop(columns=["Unnamed: 0"])

In [7]:
# Remove rows where Message is missing
df = df.dropna(subset=["Message"])

In [8]:
# Convert length column to numeric
df["length"] = pd.to_numeric(df["length"], errors="coerce")

In [9]:
# Reset index after cleaning
df = df.reset_index(drop=True)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48595 entries, 0 to 48594
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       48595 non-null  int64  
 1   Message  48595 non-null  object 
 2   length   48591 non-null  float64
 3   country  48595 non-null  object 
 4   Date     48595 non-null  object 
dtypes: float64(1), int64(1), object(3)
memory usage: 1.9+ MB


In [11]:
# Remove rows where length could not be converted
df = df.dropna(subset=["length"])

In [12]:
# Convert length to integer type
df["length"] = df["length"].astype(int)

In [13]:
# Reset index after cleaning
df = df.reset_index(drop=True)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48591 entries, 0 to 48590
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       48591 non-null  int64 
 1   Message  48591 non-null  object
 2   length   48591 non-null  int64 
 3   country  48591 non-null  object
 4   Date     48591 non-null  object
dtypes: int64(2), object(3)
memory usage: 1.9+ MB


## Text Preprocessing

In [15]:
import re

# Clean text for NLP processing
def clean_message(text):
    text = str(text).lower()                     # Convert to lowercase
    text = re.sub(r"http\S+|www\.\S+", "", text) # Remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)     # Remove punctuation (keep numbers)
    text = re.sub(r"\s+", " ", text).strip()     # Remove extra spaces
    return text

# Create cleaned text column
df["clean_text"] = df["Message"].apply(clean_message)

# Preview results
df[["Message", "clean_text"]].head(10)

,Message,clean_text
0,Bugis oso near wat...,bugis oso near wat
1,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
2,I dunno until when... Lets go learn pilates...,i dunno until when lets go learn pilates
3,Den only weekdays got special price... Haiz......,den only weekdays got special price haiz cant ...
4,Meet after lunch la...,meet after lunch la
5,m walking in citylink now ü faster come down.....,m walking in citylink now faster come down me ...
6,5 nights...We nt staying at port step liao...T...,5 nights we nt staying at port step liao too ex
7,Hey pple...$700 or $900 for 5 nights...Excelle...,hey pple 700 or 900 for 5 nights excellent loc...
8,Yun ah.the ubi one say if ü wan call by tomorr...,yun ah the ubi one say if wan call by tomorrow...
9,Hey tmr maybe can meet you at yck,hey tmr maybe can meet you at yck


## Exploratory Text Analysis

In [19]:
import re
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Simple tokenizer (no NLTK required)
def simple_tokenize(text: str):
    return re.findall(r"[a-z0-9]+", str(text).lower())

stop_words = set(ENGLISH_STOP_WORDS)

# Tokenize and remove stopwords
df["tokens"] = df["clean_text"].apply(simple_tokenize)
df["tokens_no_stop"] = df["tokens"].apply(lambda words: [w for w in words if w not in stop_words])

df[["clean_text", "tokens_no_stop"]].head()

,clean_text,tokens_no_stop
0,bugis oso near wat,"[bugis, oso, near, wat]"
1,go until jurong point crazy available only in ...,"[jurong, point, crazy, available, bugis, n, gr..."
2,i dunno until when lets go learn pilates,"[dunno, lets, learn, pilates]"
3,den only weekdays got special price haiz cant ...,"[den, weekdays, got, special, price, haiz, eat..."
4,meet after lunch la,"[meet, lunch, la]"


In [20]:
# Flatten tokens and count word frequency
all_words = [w for tokens in df["tokens_no_stop"] for w in tokens]
word_freq = Counter(all_words)

word_freq.most_common(20)

[('u', 10284),
 ('haha', 6968),
 ('s', 3495),
 ('lol', 3260),
 ('t', 2999),
 ('got', 2541),
 ('m', 2458),
 ('ok', 2440),
 ('just', 2313),
 ('time', 2034),
 ('hahaha', 1822),
 ('okay', 1741),
 ('ur', 1671),
 ('2', 1665),
 ('going', 1566),
 ('le', 1548),
 ('oh', 1517),
 ('hey', 1495),
 ('like', 1446),
 ('think', 1430)]

In [22]:
##Improve Stopword List

# Custom SMS stopwords
sms_stopwords = {
    "u", "ur", "m", "s", "t", "2", "haha", "hahaha", "lol",
    "oh", "hey", "ok", "okay", "just"
}

# Combine with sklearn stopwords
all_stopwords = stop_words.union(sms_stopwords)

# Re-filter tokens
df["tokens_no_stop"] = df["tokens"].apply(
    lambda words: [w for w in words if w not in all_stopwords]
)

# Recalculate word frequencies
all_words = [w for tokens in df["tokens_no_stop"] for w in tokens]
word_freq = Counter(all_words)

word_freq.most_common(20)

[('got', 2541),
 ('time', 2034),
 ('going', 1566),
 ('le', 1548),
 ('like', 1446),
 ('think', 1430),
 ('ll', 1429),
 ('come', 1395),
 ('yeah', 1343),
 ('home', 1322),
 ('know', 1318),
 ('good', 1313),
 ('need', 1285),
 ('later', 1245),
 ('dun', 1212),
 ('want', 1204),
 ('hi', 1159),
 ('sorry', 1153),
 ('d', 1146),
 ('meet', 1137)]

## Word Frequency Insights

After removing standard and SMS-specific stopwords, the most frequent terms suggest that SMS communication is primarily used for:

- Coordinating plans (meet, time, later, come)
- Casual social interaction (yeah, sorry, good)
- Informal conversational language

The dataset reflects highly informal and conversational English, including regional shorthand expressions.

In [23]:
#Bigram Analysis

from collections import Counter

# Create bigrams
bigrams = []

for tokens in df["tokens_no_stop"]:
    bigrams.extend(zip(tokens, tokens[1:]))

bigram_freq = Counter(bigrams)

bigram_freq.most_common(20)

[(('let', 'know'), 262),
 (('wat', 'time'), 239),
 (('good', 'morning'), 143),
 (('new', 'year'), 142),
 (('sob', 'sob'), 124),
 (('don', 'know'), 124),
 (('reach', 'home'), 116),
 (('good', 'night'), 109),
 (('o', 'o'), 108),
 (('home', 'le'), 105),
 (('happy', 'birthday'), 104),
 (('dont', 'know'), 100),
 (('x', 'x'), 95),
 (('dun', 'want'), 93),
 (('k', 'k'), 84),
 (('happy', 'new'), 82),
 (('decimal', 'pm'), 82),
 (('ya', 'lo'), 81),
 (('bus', 'stop'), 79),
 (('hi', 'andreu'), 76)]

## Bigram Analysis Insights

The most frequent two-word combinations reveal recurring conversational patterns:

- Coordination and planning phrases (e.g., "let know", "wat time", "reach home")
- Social greetings and events (e.g., "good morning", "happy birthday", "new year")
- Informal and regional SMS shorthand (e.g., "dun want", "ya lo")

This confirms that SMS communication in the dataset is primarily informal, social, and coordination-focused.